
# Multiband (8‑Band) UNet Fine‑Tuning for Glacial Lake Segmentation

This notebook trains a **UNet** on **8‑band** GeoTIFF tiles (size **512×512**) for binary segmentation of glacial lakes (1 = lake, 0 = non‑lake).  
It will:

- Randomly sample **1,000** tiles from a directory.
- Create **train/validation** splits.
- Fine‑tune UNet with **multiple backbones** (from `segmentation_models_pytorch`).
- Use a **BCE + Soft Dice** loss (commonly used for binary segmentation with class imbalance).
- Track and **log metrics** — IoU, Precision, Recall, F1, Accuracy, and Loss — **for train and validation** to a CSV.
- Save **the model with best Train IoU** (per backbone) under `OUTPUT_DIR/models/`.
- Produce a consolidated `results.csv` under `OUTPUT_DIR/logs/`.

> **Assumptions**
> - Imagery and labels live in separate directories, with **matching filenames** (e.g., `tile_0001.tif` in images and labels).
> - Imagery tiles are 8-band GeoTIFFs; labels are single-band (0/1).



## Setup: Install dependencies

This notebook uses:
- `torch`, `torchvision`, `timm`, and `segmentation_models_pytorch` (SMP)
- `rasterio` for GeoTIFF IO
- `albumentations` for augmentations (kept simple and safe for labels)
- `pandas` for logging


In [ ]:

# If running in a fresh environment, uncomment:
# %pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
# %pip install -q timm segmentation-models-pytorch rasterio albumentations pandas scikit-image



## Configuration

Set the directories for your 8-band tiles and label tiles (GeoTIFF).  
The notebook will create an `OUTPUT_DIR` with logs and models.


In [1]:

from pathlib import Path
import random

# === REQUIRED: Paths to your data ===
IMAGES_DIR = Path("C:/Users/gg/Documents/MS Data Science/Thesis/glacial_lake_detection_thesis/Training/3 Preparing train data/complete_chips_out")  # change to your imagery folder
LABELS_DIR = Path("C:/Users/gg/Documents/MS Data Science/Thesis/glacial_lake_detection_thesis/Training/3 Preparing train data/labels_out")  # change to your label folder (same filenames)

# === OUTPUTS ===
OUTPUT_DIR = Path("./training_output")  # will be created if missing
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "logs").mkdir(exist_ok=True, parents=True)
(OUTPUT_DIR / "models").mkdir(exist_ok=True, parents=True)

# === Sampling and training config ===
NUM_SAMPLES = 1000
RANDOM_SEED = 42
VAL_RATIO = 0.2  # 80/20 split
BATCH_SIZE = 2
NUM_WORKERS = 0
EPOCHS = 20
LR = 1e-3

# Backbones to try with UNet (SMP). You can edit this list.
BACKBONES = [
    "efficientnet-b0"
]
#BACKBONES = [
#    "resnet34",
#    "resnet50",
#    "efficientnet-b0",
#    "mobilenet_v2",
#]
random.seed(RANDOM_SEED)



## Discover tiles and sample 1,000

We match imagery and label tiles by filename. Only pairs that exist in both directories are eligible.  
We sample up to `NUM_SAMPLES` pairs at random (fewer if the dataset is smaller).


In [2]:

from pathlib import Path

def list_pairs(images_dir: Path, labels_dir: Path):
    img_files = {p.name: p for p in images_dir.glob("*.tif")}
    lbl_files = {p.name: p for p in labels_dir.glob("*.tif")}
    common = sorted(set(img_files.keys()) & set(lbl_files.keys()))
    pairs = [(img_files[n], lbl_files[n]) for n in common]
    return pairs

all_pairs = list_pairs(IMAGES_DIR, LABELS_DIR)
if not all_pairs:
    raise RuntimeError(f"No (image,label) .tif pairs found! Check IMAGES_DIR={IMAGES_DIR} and LABELS_DIR={LABELS_DIR}.")

print(f"Found {len(all_pairs)} paired tiles.")
random.shuffle(all_pairs)
sampled_pairs = all_pairs[:min(NUM_SAMPLES, len(all_pairs))]
len(sampled_pairs)


Found 1000 paired tiles.


1000


## Train/Validation Split


In [3]:

split_idx = int(len(sampled_pairs) * (1 - VAL_RATIO))
train_pairs = sampled_pairs[:split_idx]
val_pairs = sampled_pairs[split_idx:]
print(f"Train tiles: {len(train_pairs)}, Val tiles: {len(val_pairs)}")


Train tiles: 800, Val tiles: 200



## Dataset & Dataloader

- Reads **8‑band** imagery with `rasterio` (returns float32 tensor).
- Reads label (single band) and maps to float {0.0, 1.0}.
- **Normalization**: per-image, band-wise **min–max** to [0, 1] (robust & simple for multi-sensor data).  
  You may swap with domain-specific normalization if desired.
- Uses light augmentations on the training set (horizontal/vertical flips).


In [4]:

import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
import rasterio
import albumentations as A
from albumentations.pytorch import ToTensorV2

def read_geotiff(path: Path):
    with rasterio.open(path) as src:
        arr = src.read()  # shape: (bands, H, W)
    return arr

def minmax_per_band(x: np.ndarray, eps: float = 1e-6):
    # x: (C,H,W)
    c, h, w = x.shape
    x = x.astype(np.float32)
    for i in range(c):
        vmin = np.min(x[i])
        vmax = np.max(x[i])
        if vmax - vmin < eps:
            x[i] = 0.0
        else:
            x[i] = (x[i] - vmin) / (vmax - vmin + eps)
    return x

class LakeTiles(Dataset):
    def __init__(self, pairs, aug=None, normalize=True):
        self.pairs = pairs
        self.aug = aug
        self.normalize = normalize

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, lbl_path = self.pairs[idx]
        img = read_geotiff(img_path)  # (C,H,W)
        lbl = read_geotiff(lbl_path)  # assume (1,H,W) or (H,W)

        if lbl.ndim == 3 and lbl.shape[0] == 1:
            lbl = lbl[0]
        elif lbl.ndim == 3 and lbl.shape[0] > 1:
            lbl = lbl[0]  # take first band if label unexpectedly multi-band
        # Ensure binary (0/1)
        lbl = (lbl > 0).astype(np.float32)

        if self.normalize:
            img = minmax_per_band(img)

        # albumentations expects HWC; convert temporarily
        img_hwc = np.transpose(img, (1,2,0))
        lbl_hwc = lbl[..., None]

        if self.aug:
            transformed = self.aug(image=img_hwc, mask=lbl_hwc)
            img_hwc = transformed["image"]
            lbl_hwc = transformed["mask"]

        # Back to CHW
        if isinstance(img_hwc, np.ndarray):
            img = np.transpose(img_hwc, (2,0,1)).astype(np.float32)
            lbl = lbl_hwc[...,0].astype(np.float32)
            img = torch.from_numpy(img)
            lbl = torch.from_numpy(lbl)
        else:
            # ToTensorV2 case
            img = img_hwc.permute(2,0,1).contiguous().float()
            lbl = lbl_hwc[...,0].contiguous().float()

        return img, lbl

# Augmentations
train_aug = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    # 512x512 already; add slight shift/scale/rotate if you want
    # A.ShiftScaleRotate(shift_limit=0.02, scale_limit=0.1, rotate_limit=10, border_mode=0, p=0.3),
])

val_aug = A.Compose([])

train_ds = LakeTiles(train_pairs, aug=None, normalize=True)
val_ds = LakeTiles(val_pairs, aug=None, normalize=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, pin_memory=False)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, pin_memory=False)

len(train_ds), len(val_ds)


(800, 200)


## Loss & Metrics

**Loss:** Binary Cross Entropy with Logits (BCE) + **Soft Dice** (common in segmentation with imbalanced classes).  
**Metrics:** IoU, Precision, Recall, F1, Accuracy — computed after thresholding `sigmoid(logits) > 0.5`.


In [5]:

import torch
import torch.nn.functional as F

def dice_loss_with_logits(logits, targets, eps=1e-7):
    # logits: (B,1,H,W) or (B,H,W) ; targets: (B,H,W)
    probs = torch.sigmoid(logits)
    targets = targets.float()
    if probs.dim() == 4:
        probs = probs[:,0]  # (B,H,W)
    intersection = (probs * targets).sum(dim=(1,2))
    union = probs.sum(dim=(1,2)) + targets.sum(dim=(1,2)) + eps
    dice = (2 * intersection + eps) / union
    return 1 - dice.mean()

def bce_dice_loss(logits, targets):
    bce = F.binary_cross_entropy_with_logits(logits, targets)
    dsc = dice_loss_with_logits(logits, targets)
    return bce + dsc

def compute_metrics_from_logits(logits, targets):
    with torch.no_grad():
        probs = torch.sigmoid(logits)
        preds = (probs > 0.5).float()
        t = targets.float()

        tp = (preds * t).sum(dim=(1,2))
        tn = ((1-preds) * (1-t)).sum(dim=(1,2))
        fp = (preds * (1-t)).sum(dim=(1,2))
        fn = ((1-preds) * t).sum(dim=(1,2))

        eps = 1e-7
        precision = (tp / (tp + fp + eps)).mean().item()
        recall    = (tp / (tp + fn + eps)).mean().item()
        f1        = (2*precision*recall / (precision + recall + eps))
        iou       = (tp / (tp + fp + fn + eps)).mean().item()
        accuracy  = ((tp + tn) / (tp + tn + fp + fn + eps)).mean().item()

        return dict(iou=iou, precision=precision, recall=recall, f1=f1, accuracy=accuracy)



## Model: UNet (SMP) with multiple backbones

We adapt pretrained encoders (Imagenet) to **8 input channels** by **inflating** the first conv weights:
- Initialize new conv with `in_channels=8`.
- Copy the original 3-channel weights; for channels 3–7, use the **mean** of RGB kernels.

> If a backbone doesn't have 3‑channel weights (rare), we fall back to random init.


In [6]:

import segmentation_models_pytorch as smp
import torch.nn as nn
from copy import deepcopy

def inflate_first_conv_to_8ch(model: nn.Module):
    # Find the first conv with in_channels==3 and out_channels arbitrary
    first_conv = None
    for m in model.encoder.modules():
        if isinstance(m, nn.Conv2d) and m.in_channels == 3:
            first_conv = m
            break
    if first_conv is None:
        return model  # leave as-is

    w = first_conv.weight.data  # (out_ch, 3, k, k)
    out_ch, in_ch, kh, kw = w.shape
    new_conv = nn.Conv2d(8, out_ch, kernel_size=first_conv.kernel_size,
                         stride=first_conv.stride, padding=first_conv.padding,
                         dilation=first_conv.dilation, bias=(first_conv.bias is not None))
    with torch.no_grad():
        if in_ch == 3:
            mean_rgb = w.mean(dim=1, keepdim=True)  # (out_ch,1,k,k)
            new_w = torch.cat([w, mean_rgb.repeat(1, 5, 1, 1)], dim=1)  # (out_ch,8,k,k)
        else:
            # Fallback if encoder's first conv isn't 3ch for some reason
            new_w = torch.zeros((out_ch, 8, kh, kw))
            new_w[:, :min(8, in_ch)] = w[:, :min(8, in_ch)]
        new_conv.weight.copy_(new_w)
        if first_conv.bias is not None and new_conv.bias is not None:
            new_conv.bias.copy_(first_conv.bias.data)

    # Now replace in the actual model: find and set attribute
    replaced = False
    for name, module in model.encoder.named_modules():
        if module is first_conv:
            # Need to set it on its parent
            # We'll traverse again to locate parent by name segments
            parent = model.encoder
            parts = name.split(".")
            for p in parts[:-1]:
                parent = getattr(parent, p)
            setattr(parent, parts[-1], new_conv)
            replaced = True
            break
    return model

def make_unet(backbone: str, in_channels: int = 8, num_classes: int = 1, pretrained=True):
    # SMP expects in_channels; but for pretrained encoders we adapt first conv to 8ch
    encoder_weights = "imagenet" if pretrained else None
    model = smp.Unet(
        encoder_name=backbone,
        encoder_weights=encoder_weights,  # load pretrained on ImageNet
        in_channels=3,                     # temporarily 3 to load weights cleanly
        classes=num_classes,
    )
    if in_channels != 3 and pretrained:
        model = inflate_first_conv_to_8ch(model)
        # Patch the model's first stage to expect 8 channels at forward
        model.encoder._in_channels = in_channels  # hint to SMP internals
    elif in_channels != 3:
        # No pretrained weights ⇒ create directly with correct in_channels
        model = smp.Unet(
            encoder_name=backbone,
            encoder_weights=None,
            in_channels=in_channels,
            classes=num_classes,
        )
    return model



## Training Loop (per backbone)

For each backbone:
1. Build model with inflated first conv (8→channels).
2. Train for `EPOCHS` with **Adam** and `LR`.
3. After each epoch, evaluate on **train** and **val**; log metrics to CSV.
4. Save the **best-train-IoU** checkpoint under `OUTPUT_DIR/models/{backbone}_best_train_iou.pt`.


In [7]:

import torch
from torch import optim
import pandas as pd
from time import time
from datetime import datetime

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

scaler = torch.cuda.amp.GradScaler()
def run_one_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)
    total_loss = 0.0; n = 0
    agg = dict(iou=0., precision=0., recall=0., f1=0., accuracy=0.)
    for imgs, lbls in loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        if is_train: optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            logits = model(imgs)[:,0]
            loss = bce_dice_loss(logits, lbls)
        if is_train:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        total_loss += loss.item(); n += 1
        m = compute_metrics_from_logits(logits.detach(), lbls.detach())
        for k in agg: agg[k] += m[k]
    avg_loss = total_loss / max(1,n)
    for k in agg: agg[k] /= max(1,n)
    torch.cuda.empty_cache()  # light pressure relief
    return avg_loss, agg


results_csv = OUTPUT_DIR / "logs" / "results.csv"
if not results_csv.exists():
    pd.DataFrame(columns=[
        "timestamp","backbone","epoch","split",
        "loss","iou","precision","recall","f1","accuracy"
    ]).to_csv(results_csv, index=False)

for backbone in BACKBONES:
    print(f"\n=== Training backbone: {backbone} ===")
    model = make_unet(backbone, in_channels=8, num_classes=1, pretrained=True).to(device)
    optimizer = optim.Adam(model.parameters(), lr=LR)

    best_train_iou = -1.0
    best_path = OUTPUT_DIR / "models" / f"{backbone}_best_train_iou.pt"

    for epoch in range(1, EPOCHS+1):
        t0 = time()
        train_loss, train_metrics = run_one_epoch(model, train_loader, optimizer)
        val_loss,   val_metrics   = run_one_epoch(model, val_loader, optimizer=None)
        dt = time()-t0

        print(f"[{backbone}] Epoch {epoch}/{EPOCHS} — "
              f"train IoU={train_metrics['iou']:.4f}, val IoU={val_metrics['iou']:.4f}, "
              f"train Loss={train_loss:.4f}, val Loss={val_loss:.4f} ({dt:.1f}s)")

        # Log to CSV (both splits)
        ts = datetime.utcnow().isoformat()
        row_train = dict(timestamp=ts, backbone=backbone, epoch=epoch, split="train",
                         loss=train_loss, **train_metrics)
        row_val   = dict(timestamp=ts, backbone=backbone, epoch=epoch, split="val",
                         loss=val_loss, **val_metrics)
        df = pd.DataFrame([row_train, row_val])
        df.to_csv(results_csv, mode="a", index=False, header=False)

        # Save best (by TRAIN IoU as requested)
        if train_metrics["iou"] > best_train_iou:
            best_train_iou = train_metrics["iou"]
            torch.save({
                "backbone": backbone,
                "epoch": epoch,
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "best_train_iou": best_train_iou,
            }, best_path)

print(f"\nTraining complete. Logs at: {results_csv}")
print(f"Best models saved (per backbone) under: {OUTPUT_DIR / 'models'}")


Device: cuda

=== Training backbone: efficientnet-b0 ===


C:\Users\gg\AppData\Local\Temp\ipykernel_11516\2263433220.py:10: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
C:\Users\gg\AppData\Local\Temp\ipykernel_11516\2263433220.py:19: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


[efficientnet-b0] Epoch 1/20 — train IoU=0.0105, val IoU=0.0768, train Loss=0.9760, val Loss=0.9978 (98.7s)


C:\Users\gg\AppData\Local\Temp\ipykernel_11516\2263433220.py:61: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts = datetime.utcnow().isoformat()


[efficientnet-b0] Epoch 2/20 — train IoU=0.1967, val IoU=0.1810, train Loss=0.8608, val Loss=0.8648 (96.9s)
[efficientnet-b0] Epoch 3/20 — train IoU=0.2236, val IoU=0.1683, train Loss=0.8087, val Loss=1.0465 (101.9s)
[efficientnet-b0] Epoch 4/20 — train IoU=0.2408, val IoU=0.2069, train Loss=0.7448, val Loss=0.7717 (107.6s)
[efficientnet-b0] Epoch 5/20 — train IoU=0.2555, val IoU=0.2051, train Loss=0.7262, val Loss=0.7774 (107.1s)
[efficientnet-b0] Epoch 6/20 — train IoU=0.2644, val IoU=0.2230, train Loss=0.7161, val Loss=0.7596 (112.4s)
[efficientnet-b0] Epoch 7/20 — train IoU=0.2672, val IoU=0.2224, train Loss=0.7125, val Loss=0.7618 (109.7s)
[efficientnet-b0] Epoch 8/20 — train IoU=0.2732, val IoU=0.2091, train Loss=0.7044, val Loss=0.7720 (111.6s)
[efficientnet-b0] Epoch 9/20 — train IoU=0.2813, val IoU=0.2337, train Loss=0.6939, val Loss=0.7609 (108.1s)
[efficientnet-b0] Epoch 10/20 — train IoU=0.2863, val IoU=0.2452, train Loss=0.6885, val Loss=0.7284 (111.5s)
[efficientnet-b0] E


## (Optional) Summarize Results

Quickly summarize the best validation IoU per backbone from the CSV logs.


In [ ]:

import pandas as pd

df = pd.read_csv(OUTPUT_DIR / "logs" / "results.csv")
best_val = (df[df.split=="val"]
            .sort_values(["backbone","iou"], ascending=[True,False])
            .groupby("backbone").head(1)
            .reset_index(drop=True))
best_val
